In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("pipeline_run_id", "")

pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

import uuid
from datetime import datetime

from pyspark.sql.functions import (
    col,
    lit,
    upper,
    trim,
    lower,
    upper as spark_upper,
    current_timestamp,
    regexp_replace
)

NOTEBOOK_NAME = "Enterprise_Generic_Silver_Loader_v4"

print("=" * 80)
print(NOTEBOOK_NAME)
print("=" * 80)
print("Pipeline Run :", pipeline_run_id)

Enterprise_Generic_Silver_Loader_v4
Pipeline Run : 


In [0]:
def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def print_info(name, value):
    print(f"{name:<30}: {value}")


def print_success(message):
    print(f"SUCCESS : {message}")


def print_warning(message):
    print(f"WARNING : {message}")


def print_error(message):
    print(f"ERROR   : {message}")

In [0]:

print_header("READING ACTIVE METADATA")

metadata_df = (
    spark.table(f"{catalog_name}.{metadata_schema}.bronze_config")
         .filter(upper(col("active")) == "Y")
         .orderBy("table_name")
)

metadata_rows = metadata_df.collect()

if len(metadata_rows) == 0:
    raise Exception("No active metadata found.")

print_success(f"{len(metadata_rows)} table(s) found.")

for r in metadata_rows:
    print_info("Table", r["table_name"])


READING ACTIVE METADATA
SUCCESS : 5 table(s) found.
Table                         : Agent
Table                         : Branch
Table                         : Claim
Table                         : Customer
Table                         : Policy


In [0]:
# COMMAND ----------
print_header("PROCESSING ALL ACTIVE TABLES")
results_summary = []

for config in metadata_rows:
    table_name = config["table_name"]
    load_id = str(uuid.uuid4())
    print_header(f"PROCESSING {table_name}")

    try:
        # ----------------------------
        # Metadata
        # ----------------------------
        target_table = config["target_table"]
        primary_key = config["primary_key"]
        compare_columns = config["compare_columns"]
        source_system = config["source_system"]
        load_strategy = config["load_strategy"]
        gold_table = config["gold_table"]
        history_table = config["history_table"]

        bronze_table = (f"{catalog_name}.{bronze_schema}.{target_table}")
        silver_table = (f"{catalog_name}.{silver_schema}.{target_table}")

        print_info("Bronze Table", bronze_table)
        print_info("Silver Table", silver_table)

        # ----------------------------
        # Read Bronze
        # ----------------------------
        df = spark.table(bronze_table)
        rows_before = df.count()
        print_info("Rows Read", rows_before)

        # ----------------------------
        # Generic Cleansing
        # ----------------------------
        print_header("DATA CLEANSING")

        # Trim every string column
        for field in df.schema.fields:
            if field.dataType.simpleString() == "string":
                df = df.withColumn(
                    field.name,
                    trim(col(field.name))
                )

        # Remove duplicate primary keys
        df = df.dropDuplicates([primary_key])

        duplicate_count = get_duplicate_count(df, primary_key)
        null_key_count = get_null_count(df, primary_key)

        print_info("Duplicate Keys", duplicate_count)
        print_info("Null Keys", null_key_count)

        # ----------------------------
        # Standardization
        # ----------------------------
        print_header("STANDARDIZATION")

        string_columns = [
            f.name
            for f in df.schema.fields
            if f.dataType.simpleString() == "string"
        ]

        for c in string_columns:
            df = df.withColumn(
                c,
                regexp_replace(
                    trim(col(c)),
                    "\\s+",
                    " "
                )
            )

        # Email
        if "email" in df.columns:
            df = df.withColumn("email", lower(col("email")))

        # Gender
        if "gender" in df.columns:
            df = df.withColumn("gender", upper(col("gender")))

        # Country
        if "country" in df.columns:
            df = df.withColumn("country", upper(col("country")))

        print_success("Standardization Completed")

        # ----------------------------
        # Business Validation
        # ----------------------------
        print_header("BUSINESS VALIDATION")

        if "email" in df.columns:
            df = df.filter(col("email").isNotNull())

        if primary_key in df.columns:
            df = df.filter(col(primary_key).isNotNull())

        rows_after = df.count()
        print_info("Rows After Validation", rows_after)

        # ----------------------------
        # Metadata Columns
        # ----------------------------
        df = add_metadata(
            df=df,
            source_system=source_system,
            pipeline_run_id=pipeline_run_id,
            load_id=load_id
        )

        # ----------------------------
        # Write Silver
        # ----------------------------
        print_header("WRITING SILVER")

        writer = (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )

        writer.saveAsTable(silver_table)

        rows_written = rows_after
        silver_total_rows = spark.table(silver_table).count()

        print_success("Silver Load Completed")
        print_info("Rows Written", rows_written)
        print_info("Silver Total Rows", silver_total_rows)

        # ----------------------------
        # Audit (success)
        # ----------------------------
        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=target_table,
            load_type=load_strategy,
            rows_read=rows_before,
            rows_written=rows_written,
            status="SUCCESS",
            error_message="",
            pipeline_run_id=pipeline_run_id
        )

        results_summary.append({
            "table_name": table_name,
            "load_id": load_id,
            "status": "SUCCESS",
            "rows_read": rows_before,
            "rows_written": rows_written,
            "error": None
        })

    except Exception as e:
        # Handle errors gracefully so one failing table doesn't crash the entire loop
        print_error(f"Failed to process table {table_name}: {str(e)}")

        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=table_name,
            load_type=config["load_strategy"] if config["load_strategy"] else "UNKNOWN",
            rows_read=0,
            rows_written=0,
            status="FAILED",
            error_message=str(e),
            pipeline_run_id=pipeline_run_id
        )

        results_summary.append({
            "table_name": table_name,
            "load_id": load_id,
            "status": "FAILED",
            "rows_read": 0,
            "rows_written": 0,
            "error": str(e)
        })

# Print summary after the loop completes
print_header("PROCESSING SUMMARY")
for result in results_summary:
    print(f"Table: {result['table_name']} | Status: {result['status']} | Rows: {result['rows_read']}")



PROCESSING ALL ACTIVE TABLES

PROCESSING Agent
Bronze Table                  : dbw_insurance.insurance_bronze.agent
Silver Table                  : dbw_insurance.insurance_silver.agent
Rows Read                     : 2005

DATA CLEANSING
Duplicate Keys                : 0
Null Keys                     : 0

STANDARDIZATION
SUCCESS : Standardization Completed

BUSINESS VALIDATION
Rows After Validation         : 1005

WRITING SILVER
SUCCESS : Silver Load Completed
Rows Written                  : 1005
Silver Total Rows             : 1005

PROCESSING Branch
Bronze Table                  : dbw_insurance.insurance_bronze.branch
Silver Table                  : dbw_insurance.insurance_silver.branch
Rows Read                     : 1000

DATA CLEANSING
Duplicate Keys                : 0
Null Keys                     : 0

STANDARDIZATION
SUCCESS : Standardization Completed

BUSINESS VALIDATION
Rows After Validation         : 1000

WRITING SILVER
SUCCESS : Silver Load Completed
Rows Written         

In [0]:
# COMMAND ----------

print("\n")
print("=" * 90)
print("ENTERPRISE GENERIC SILVER LOADER V4 COMPLETED")
print("=" * 90)

for r in results_summary:
    line = (
        f"{r['table_name']:<15}"
        f": {r['status']:<8}"
        f" Read={r['rows_read']:<6}"
        f" Written={r['rows_written']:<6}"
    )

    if r["status"] == "FAILED":
        line += f" ERROR: {r['error']}"
    print(line)
success_count = len(
    [
        x

        for x in results_summary
        if x["status"] == "SUCCESS"
    ]
)

failed_count = len(
    [
        x
        for x in results_summary
        if x["status"] == "FAILED"
    ]
)

print("=" * 90)
print_success(
    f"Processed {len(results_summary)} tables."
    f" Success={success_count}"
    f" Failed={failed_count}"
)

print("=" * 90)



ENTERPRISE GENERIC SILVER LOADER V4 COMPLETED
Agent          : SUCCESS  Read=2005   Written=1005  
Branch         : SUCCESS  Read=1000   Written=1000  
Claim          : SUCCESS  Read=943    Written=943   
Customer       : SUCCESS  Read=34170  Written=925   
Policy         : SUCCESS  Read=1000   Written=1000  
SUCCESS : Processed 5 tables. Success=5 Failed=0
